# Unsloth Qwen3 SFT/RL Notebook


## 0. 指定 GPU

In [ ]:
import os
import torch

GPU_ID = "2"  # 改成 nvidia-smi 里空闲的物理 GPU 编号

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("Visible GPU count:", torch.cuda.device_count())
print("Current CUDA device:", torch.cuda.current_device())
print("GPU name:", torch.cuda.get_device_name(0))

free, total = torch.cuda.mem_get_info(0)
print("Free GB:", round(free / 1024**3, 2))
print("Total GB:", round(total / 1024**3, 2))


## 1. 实验配置


In [ ]:
from pathlib import Path
import sys

# 自动识别仓库根目录：既支持从 repo root 打开，也支持从 notebooks/ 打开。
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

# ========== file path ==========
MODEL_NAME_OR_PATH = str(REPO_ROOT / "models" / "Qwen3-4B-Instruct-2507")
TRAIN_FILE = REPO_ROOT / "data" / "split_data" / "toy_train.json"  # 原始训练集 JSON 文件
VALID_FILE = REPO_ROOT / "data" / "split_data" / "toy_valid.json"  # 原始验证集 JSON 文件；用于每个 epoch 的 ELM 指标评估
TEST_FILE = REPO_ROOT / "data" / "split_data" / "toy_test.json"    # 原始测试集 JSON 文件；不参与训练过程
PROCESSED_TRAIN_FILE = REPO_ROOT / "data" / "processed_data" / "processed_train_messages.json"
PROCESSED_VALID_FILE = REPO_ROOT / "data" / "processed_data" / "processed_valid_messages.json"
PROCESSED_TEST_FILE = REPO_ROOT / "data" / "processed_data" / "processed_test_messages.json"
SFT_OUTPUT_DIR = REPO_ROOT / "outputs" / "lora_files" / "qwen3_4b_unsloth_lora"
ELM_EVAL_DIR = SFT_OUTPUT_DIR / "elm_eval_by_epoch"
BEST_ELM_CHECKPOINT_DIR = SFT_OUTPUT_DIR / "best_elm_checkpoint"
PRED_OUTPUT_FILE = REPO_ROOT / "outputs" / "predictions" / "qwen3_4b_test_predictions.json"

# 你的原始数据字段：processed JSON 只保留这些关键字段，并额外写入标准 messages。
PROMPT_FIELD = "prompt"
RESPONSE_FIELD = "groundtruth"
RESPONSE_FIELD_CANDIDATES = ["groundtruth", "response", "answer", "label", "output", "completion"]
ID_FIELD = "prompt_id"
CLAIM_ID_FIELD = "claim_id"
CONDITION_FIELD = "condition"
PROCESSED_KEEP_FIELDS = [ID_FIELD, CLAIM_ID_FIELD, PROMPT_FIELD, RESPONSE_FIELD, CONDITION_FIELD]
SYSTEM_PROMPT = "You are a respondent in a persuasion scenario. Answer from the assigned role's perspective."
ASSISTANT_ROLE = "assistant"  # 不建议改成 respondent；多数 chat template 只支持 assistant。
ENABLE_THINKING = False      # Qwen3 默认可能插入 <think>...</think>；SFT/量表任务建议关闭。

# 模型 / QLoRA 设置
MAX_SEQ_LENGTH = 512       # 如果 OOM，先降到 512
LOAD_IN_4BIT = True         # True=QLoRA；False=普通 LoRA
DTYPE = None                # None 让 Unsloth 自动选择

# LoRA 超参
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# SFT 超参
RESPONSE_ONLY_LOSS = True   # True=只训练 groundtruth/assistant answer；False=prompt+answer 都算 loss
NUM_TRAIN_EPOCHS = 8
LEARNING_RATE = 2e-4
BATCH_SIZE = 8
GRAD_ACCUM = 4

# 批量推理参数
MAX_NEW_TOKENS = 32         # 你的任务只需要输出 Likert 句子，32/64 通常足够
TEMPERATURE = 0.0           # 量表预测建议先用 0，稳定可复现
TOP_P = 1.0
INFER_BATCH_SIZE = 100      # decoder-only batch generation 会使用 left padding
NUM_REPEATS = 1
MIN_FREE_GPU_MEMORY_GB = 6.0

print("Repo root:", REPO_ROOT)
print("Model path:", MODEL_NAME_OR_PATH)
print("Train file:", TRAIN_FILE)
print("Valid file:", VALID_FILE)
print("Test file:", TEST_FILE)
print("Processed train:", PROCESSED_TRAIN_FILE)
print("Processed valid:", PROCESSED_VALID_FILE)
print("Processed test:", PROCESSED_TEST_FILE)
print("ELM eval dir:", ELM_EVAL_DIR)
print("Best ELM checkpoint dir:", BEST_ELM_CHECKPOINT_DIR)
print("Model exists:", Path(MODEL_NAME_OR_PATH).exists())
print("Train exists:", TRAIN_FILE.exists())
print("Valid exists:", VALID_FILE.exists())
print("Test exists:", TEST_FILE.exists())


## 2. 导入依赖并检查 CUDA

In [ ]:
import inspect
import json
import re
from typing import Any

import torch
from unsloth import FastLanguageModel, is_bfloat16_supported

from llm_lab.data import _apply_chat_template, load_sft_dataset, load_grpo_dataset
from llm_lab.elm_eval import (
    EpochELMEvalCallback,
    batch_generate_predictions,
    compute_elm_stats,
    load_json,
    save_json,
)
from llm_lab.model_utils import ensure_pad_token
from llm_lab.train_utils import ResponseOnlyDataCollator, build_sft_trainer, get_training_args


## 3. 加载模型并注入 LoRA

这部分基本等同于 Unsloth 官方文档里的核心代码。

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME_OR_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)

def disable_thinking_by_default(tokenizer, enable_thinking: bool = False):
    """Make Qwen3 chat templates default to enable_thinking=False.
    """
    apply_chat_template = getattr(tokenizer, "apply_chat_template", None)
    if apply_chat_template is None:
        return tokenizer
    try:
        parameters = inspect.signature(apply_chat_template).parameters
    except (TypeError, ValueError):
        return tokenizer
    if "enable_thinking" not in parameters:
        return tokenizer

    def apply_chat_template_without_thinking(*args, **kwargs):
        kwargs.setdefault("enable_thinking", enable_thinking)
        return apply_chat_template(*args, **kwargs)

    tokenizer.apply_chat_template = apply_chat_template_without_thinking
    return tokenizer


ensure_pad_token(tokenizer)
tokenizer = disable_thinking_by_default(tokenizer, ENABLE_THINKING)
tokenizer.padding_side = "left"
print(f"Qwen thinking enabled: {ENABLE_THINKING}")

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 4. 读取并标准化 train/test 两个 JSON 文件

这里不再依赖 `split` 字段。Notebook 会把原始字段数据转成标准 `messages` 格式：

- train：`system + user(prompt) + assistant(groundtruth)`，用于 SFT。
- test：`system + user(prompt)`；如果 test 里也有 `groundtruth`，会保留下来用于评估。

如果你的答案字段不叫 `groundtruth`，优先在配置区修改 `RESPONSE_FIELD_CANDIDATES`。


In [ ]:
def read_json_or_jsonl(path: str | Path) -> list[dict[str, Any]]:
    path = Path(path)
    raw = path.read_text(encoding="utf-8").strip()
    if not raw:
        raise ValueError(f"Empty data file: {path}")
    if path.suffix.lower() == ".json" or raw.startswith("["):
        rows = json.loads(raw)
    else:
        rows = [json.loads(line) for line in raw.splitlines() if line.strip()]
    if not isinstance(rows, list) or not all(isinstance(row, dict) for row in rows):
        raise ValueError("Data must be a JSON array or JSONL of objects.")
    return rows


def first_text_field(row: dict[str, Any], candidates: list[str]) -> tuple[str | None, str | None]:
    for field in candidates:
        value = row.get(field)
        if isinstance(value, str) and value.strip():
            return field, value.strip()
    return None, None


def build_messages(prompt: str, response: str | None = None) -> list[dict[str, str]]:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    if response is not None:
        messages.append({"role": ASSISTANT_ROLE, "content": response})
    return messages


def compact_row(row: dict[str, Any], require_response: bool) -> dict[str, Any]:
    prompt = row.get(PROMPT_FIELD)
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError(f"missing non-empty prompt field {PROMPT_FIELD!r}")

    response_field, response = first_text_field(row, RESPONSE_FIELD_CANDIDATES)
    if require_response and response is None:
        raise ValueError(
            "missing answer field; tried "
            f"{RESPONSE_FIELD_CANDIDATES}. Add your real answer field to RESPONSE_FIELD_CANDIDATES."
        )

    compact: dict[str, Any] = {}
    for field in PROCESSED_KEEP_FIELDS:
        if field == RESPONSE_FIELD:
            if response is not None:
                compact[RESPONSE_FIELD] = response
        elif field in row:
            compact[field] = row[field]

    # 保证训练、验证、测试最关键字段一定存在，并统一答案字段名为 groundtruth。
    compact[PROMPT_FIELD] = prompt.strip()
    if response is not None:
        compact[RESPONSE_FIELD] = response
    compact["messages"] = build_messages(prompt.strip(), response)
    return compact


def standardize_rows(rows: list[dict[str, Any]], name: str, require_response: bool) -> list[dict[str, Any]]:
    if not rows:
        raise ValueError(f"{name} is empty")
    converted = []
    for idx, row in enumerate(rows):
        try:
            converted.append(compact_row(row, require_response=require_response))
        except ValueError as exc:
            raise ValueError(f"{name} row {idx} cannot be converted: {exc}") from exc
    return converted


train_raw_rows = read_json_or_jsonl(TRAIN_FILE)
valid_raw_rows = read_json_or_jsonl(VALID_FILE)
test_raw_rows = read_json_or_jsonl(TEST_FILE) if TEST_FILE.exists() else []

train_rows = standardize_rows(train_raw_rows, "train", require_response=True)
valid_rows = standardize_rows(valid_raw_rows, "valid", require_response=True)
test_rows = standardize_rows(test_raw_rows, "test", require_response=False) if test_raw_rows else []

PROCESSED_TRAIN_FILE.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_VALID_FILE.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_TEST_FILE.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_TRAIN_FILE.write_text(json.dumps(train_rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
PROCESSED_VALID_FILE.write_text(json.dumps(valid_rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
PROCESSED_TEST_FILE.write_text(json.dumps(test_rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print(f"Train rows: {len(train_rows)} from {TRAIN_FILE}")
print(f"Valid rows: {len(valid_rows)} from {VALID_FILE}")
print(f"Test rows : {len(test_rows)} from {TEST_FILE}")
print(f"Wrote compact processed train data to: {PROCESSED_TRAIN_FILE}")
print(f"Wrote compact processed valid data to: {PROCESSED_VALID_FILE}")
print(f"Wrote compact processed test data to : {PROCESSED_TEST_FILE}")
print("Train columns:", sorted(train_rows[0].keys()))
print("Valid columns:", sorted(valid_rows[0].keys()))
if test_rows:
    print("Test columns:", sorted(test_rows[0].keys()))
print("Example processed train row:")
print(json.dumps(train_rows[0], ensure_ascii=False, indent=2)[:1200])


## 5. 构造 SFT Dataset

这里训练不直接吃原始 JSON，而是吃上一步写出的 `PROCESSED_TRAIN_FILE`：

- processed train/valid/test 只保留关键字段，并额外包含标准 messages：`prompt_id`、`claim_id`、`prompt`、`groundtruth`、`condition`。
- `prompt` 会作为 user 输入。
- `groundtruth` 会作为 assistant answer。
- `RESPONSE_ONLY_LOSS=True` 时，loss 只算 assistant answer 部分，不会训练 prompt。


In [ ]:
# 上面已经 monkeypatch tokenizer.apply_chat_template 默认关闭 thinking。
train_dataset = load_sft_dataset(
    PROCESSED_TRAIN_FILE,
    tokenizer,
    response_only_loss = RESPONSE_ONLY_LOSS,
)

print(train_dataset)
print("Columns:", train_dataset.column_names)
print("First row keys:", train_dataset[0].keys())


In [ ]:
train_dataset[0]["text"]

## 6. SFT 训练

- `RESPONSE_ONLY_LOSS=True`：使用 `Trainer + ResponseOnlyDataCollator`，只对 `groundtruth` 计算 loss。
- `RESPONSE_ONLY_LOSS=False`：使用 TRL `SFTTrainer`，对完整 prompt+answer 计算 loss。

In [ ]:
from transformers import Trainer

fp16 = not is_bfloat16_supported()
bf16 = is_bfloat16_supported()

training_args = get_training_args(
    output_dir = str(SFT_OUTPUT_DIR),
    max_length = MAX_SEQ_LENGTH,
    num_train_epochs = NUM_TRAIN_EPOCHS,
    learning_rate = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    fp16 = fp16,
    bf16 = bf16,
)

if RESPONSE_ONLY_LOSS:
    trainer = Trainer(
        model = model,
        args = training_args,
        train_dataset = train_dataset,
        data_collator = ResponseOnlyDataCollator(tokenizer, max_length=MAX_SEQ_LENGTH),
    )
else:
    trainer = build_sft_trainer(model, tokenizer, train_dataset, training_args)

elm_eval_callback = EpochELMEvalCallback(
    tokenizer = tokenizer,
    train_rows = train_rows,
    valid_rows = valid_rows,
    output_dir = ELM_EVAL_DIR,
    best_checkpoint_dir = BEST_ELM_CHECKPOINT_DIR,
    system_prompt = SYSTEM_PROMPT,
    keep_fields = PROCESSED_KEEP_FIELDS,
    prompt_field = PROMPT_FIELD,
    response_field = RESPONSE_FIELD,
    max_new_tokens = MAX_NEW_TOKENS,
    temperature = TEMPERATURE,
    top_p = TOP_P,
    batch_size = INFER_BATCH_SIZE,
    num_repeats = NUM_REPEATS,
    device = "cuda",
)
trainer.add_callback(elm_eval_callback)

trainer.train()


## 7. 保存 LoRA adapter / 合并模型

In [ ]:
SFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(SFT_OUTPUT_DIR))
tokenizer.save_pretrained(str(SFT_OUTPUT_DIR))
print("Saved LoRA adapter to:", SFT_OUTPUT_DIR)

# 如果需要部署合并模型，取消下面一行注释：
# trainer.model.save_pretrained_merged(str(SFT_OUTPUT_DIR) + "_merged_16bit", tokenizer, save_method="merged_16bit")

## 8. 单条推理 sanity check

从测试集拿一条 prompt 看模型输出格式是否正确。

In [ ]:
def generate_one(row: dict[str, Any]) -> str:
    pred = batch_generate_predictions(
        trainer.model,
        tokenizer,
        [row],
        system_prompt=SYSTEM_PROMPT,
        keep_fields=PROCESSED_KEEP_FIELDS,
        prompt_field=PROMPT_FIELD,
        response_field=RESPONSE_FIELD,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        batch_size=1,
        num_repeats=1,
        device="cuda",
        progress_prefix="sample",
    )[0]
    return pred["model_output"]

sample = test_rows[0] if test_rows else valid_rows[0]
print("Prompt id:", sample.get(ID_FIELD))
print("Groundtruth:", sample.get(RESPONSE_FIELD))
print("Model output:", generate_one(sample))


## 9. 测试集批量推理并保存结果

输出文件会保留你的原始字段（如 `prompt_id`、`claim`、`condition`、`groundtruth` 等），并新增：

- `model_output`：模型完整输出。
- `parsed_score`：从输出中抽取的 1-11 分数；解析失败则为 `None`。
- `is_exact_match`：模型输出和 `groundtruth` 去空格后是否完全一致。

In [ ]:
pred_rows = batch_generate_predictions(
    trainer.model,
    tokenizer,
    test_rows,
    system_prompt=SYSTEM_PROMPT,
    keep_fields=PROCESSED_KEEP_FIELDS,
    prompt_field=PROMPT_FIELD,
    response_field=RESPONSE_FIELD,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    batch_size=INFER_BATCH_SIZE,
    num_repeats=NUM_REPEATS,
    device="cuda",
    progress_prefix="test",
)
PRED_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
PRED_OUTPUT_FILE.write_text(json.dumps(pred_rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {len(pred_rows)} predictions to {PRED_OUTPUT_FILE}")
if pred_rows:
    print(json.dumps({k: pred_rows[0].get(k) for k in [ID_FIELD, RESPONSE_FIELD, "model_output", "parsed_score", "is_exact_match"]}, ensure_ascii=False, indent=2))


In [ ]:
import csv
from pathlib import Path

pred_path = Path(PRED_OUTPUT_FILE)
ELM_STATS_FILE = pred_path.with_name(pred_path.stem + "_elm_stats.json")
ELM_CLAIM_STATS_CSV = pred_path.with_name(pred_path.stem + "_claim_stats.csv")

data = load_json(pred_path)
result = compute_elm_stats(data, prediction_file=pred_path)
save_json(result, ELM_STATS_FILE)

# 同时导出一个 claim-level CSV，方便后续看表
with open(ELM_CLAIM_STATS_CSV, "w", encoding="utf-8", newline="") as f:
    fieldnames = list(result["claim_stats"][0].keys())
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(result["claim_stats"])

print("ELM statistics saved to:", ELM_STATS_FILE)
print("Claim-level CSV saved to:", ELM_CLAIM_STATS_CSV)
print()
print(json.dumps(result["summary"], ensure_ascii=False, indent=2))


## 10. 可选：GRPO / RL

默认不运行。你要做 RL 时再取消注释，并把 reward 函数替换成你的任务 reward。例如你的任务可以用 `parsed_score` 与目标分数的距离设计 reward。

```python
# from unsloth import PatchFastRL
# PatchFastRL("grpo", FastLanguageModel)
# from trl import GRPOConfig, GRPOTrainer
# rl_dataset = load_grpo_dataset(PROCESSED_TRAIN_FILE, answer_field=RESPONSE_FIELD)
# ...
```


## 11. 对应脚本命令

Notebook 会先把原始数据写成 `data/processed_data/processed_train_messages.json` 和 `data/processed_data/processed_test_messages.json`。Notebook 跑通后，正式训练可以直接用 processed 文件：

```bash
CUDA_VISIBLE_DEVICES=2 python scripts/train_lora.py \
  --model_name_or_path models/Qwen3-1.7B \
  --train_file data/processed_data/processed_train_messages.json \
  --output_dir outputs/lora_files/qwen3_1.7b_unsloth_lora

CUDA_VISIBLE_DEVICES=2 python scripts/batch_infer_lora.py \
  --model_name_or_path outputs/lora_files/qwen3_1.7b_unsloth_lora \
  --input_file data/processed_data/processed_test_messages.json \
  --output_file outputs/predictions/qwen3_1.7b_test_predictions.json \
  --overwrite
```


## Post-SFT custom REINFORCE RL

This section keeps the SFT flow unchanged and runs a separate claim-group REINFORCE loop after SFT. Set `RL_MODEL_NAME_OR_PATH` to either the original Qwen3-4B path or an SFT merged model directory.


In [ ]:
from pathlib import Path
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm_lab.model_utils import ensure_pad_token
from llm_lab.rl_reinforce import RLConfig, load_groups_from_file, reinforce_train, save_config

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

RL_MODEL_NAME_OR_PATH = str(REPO_ROOT / "models" / "Qwen3-4B-Instruct-2507")  # or outputs/.../merged_sft_model
RL_TRAIN_FILE = REPO_ROOT / "data" / "processed_data" / "processed_train_messages.json"
RL_OUTPUT_DIR = REPO_ROOT / "outputs" / "lora_files_rl" / "qwen3_4b_elm_reinforce"

rl_cfg = RLConfig(
    model_name_or_path=RL_MODEL_NAME_OR_PATH,
    train_file=str(RL_TRAIN_FILE),
    output_dir=str(RL_OUTPUT_DIR),
    num_train_epochs=1,
    groups_per_step=1,
    rollouts_per_group=2,
    max_new_tokens=32,
    temperature=0.7,
    top_p=0.9,
    verbose=True,
)
save_config(rl_cfg)
print("RL config saved to:", RL_OUTPUT_DIR / "rl_config.json", flush=True)

print("Loading RL tokenizer from:", rl_cfg.model_name_or_path, flush=True)
rl_tokenizer = AutoTokenizer.from_pretrained(rl_cfg.model_name_or_path, trust_remote_code=True)
ensure_pad_token(rl_tokenizer)
rl_tokenizer.padding_side = 'left'

print("Loading RL policy model; this can take several minutes for Qwen3-4B...", flush=True)
rl_model = AutoModelForCausalLM.from_pretrained(
    rl_cfg.model_name_or_path,
    trust_remote_code=True,
    torch_dtype='auto',
    device_map='auto',
)
print("RL policy model loaded.", flush=True)
rl_model.gradient_checkpointing_enable()
rl_model.config.use_cache = False

print("Loading complete claim groups from:", rl_cfg.train_file, flush=True)
rl_groups = load_groups_from_file(rl_cfg.train_file)
print(f"Loaded {len(rl_groups)} complete claim groups.", flush=True)

rl_history = reinforce_train(rl_model, rl_tokenizer, rl_groups, rl_cfg)
print("Saving final RL checkpoint...", flush=True)
rl_model.save_pretrained(RL_OUTPUT_DIR / 'final_checkpoint')
rl_tokenizer.save_pretrained(RL_OUTPUT_DIR / 'final_checkpoint')
print("Done. Final checkpoint:", RL_OUTPUT_DIR / 'final_checkpoint', flush=True)
